# SFT DataLoader：Non-greedy 与 Greedy Packing

Attention 能否正确隔离样本，取决于 DataLoader 把哪些样本放进同一条训练序列，以及它是否保留了每条样本的起点。本节从同一组 Wordle 样本出发，对比两种 packing 策略，并定义 Attention backend 必须遵守的输入契约。

**前置知识**：tokenization、EOS、causal Attention、`seq_len`。

**学习目标**：

- 计算 non-greedy 与 greedy packing 的 token-slot 利用率；
- 解释每条样本重新从 0 编号、labels 和 attention mask 的不同职责；
- 判断一种 DataLoader 输出需要 causal 还是 document-aware Attention；
- 看懂当前实现如何根据 attention 能力选择 packing 路线。


## 1. 从 raw sample 到训练 tensor

Wordle 原始记录先经过 `chat_processor` 整理为消息，再由 chat template 和 tokenizer 转为 tokens。DataLoader 随后构造 shifted labels，并决定多条 tokenized samples 是否共享一个固定长度 container。

```text
raw Wordle sample
  → chat_processor
  → chat template + tokenizer
  → input_ids / labels
  → non-greedy 或 greedy packing
  → input + positions + labels
```


## 2. Non-greedy：每条样本保持独立

Non-greedy 路线不把不同样本放入同一个 sequence。若训练 shape 固定为 `seq_len=1024`，每条短样本都需要 padding 到 1024。普通 causal mask 在这里是正确的，因为一个 sequence 中没有第二个文档。


In [ ]:
from __future__ import annotations

SEQ_LEN = 1024
lengths = [180, 260, 410, 120]

def non_greedy_layout(sample_lengths: list[int], capacity: int) -> list[list[int]]:
    if any(length > capacity for length in sample_lengths):
        raise ValueError('sample longer than sequence capacity')
    return [[length] for length in sample_lengths]

non_greedy = non_greedy_layout(lengths, SEQ_LEN)
non_greedy


In [ ]:
def layout_stats(containers: list[list[int]], capacity: int) -> dict[str, float]:
    useful = sum(sum(container) for container in containers)
    slots = len(containers) * capacity
    return {
        'raw_samples': sum(len(container) for container in containers),
        'containers': len(containers),
        'useful_tokens': useful,
        'allocated_slots': slots,
        'utilization': useful / slots,
    }

layout_stats(non_greedy, SEQ_LEN)


## 3. Greedy packing：让短样本共享 container

Greedy packing 按数据顺序把样本加入当前 container；下一条放不下时，先用 padding 补齐当前 container，再开启新的 container。它不改变每条样本自身的 token 顺序。


In [ ]:
def greedy_pack(sample_lengths: list[int], capacity: int) -> list[list[int]]:
    containers: list[list[int]] = []
    current: list[int] = []
    used = 0

    for length in sample_lengths:
        if length > capacity:
            raise ValueError(f'sample length {length} exceeds capacity {capacity}')
        if current and used + length > capacity:
            containers.append(current)
            current, used = [], 0
        current.append(length)
        used += length

    if current:
        containers.append(current)
    return containers

greedy = greedy_pack(lengths, SEQ_LEN)
print('non-greedy:', non_greedy)
print('greedy:    ', greedy)
print('greedy stats:', layout_stats(greedy, SEQ_LEN))


这里四条样本共 970 tokens，可以放入同一个 1024-token container。token-slot 利用率显著提高，但 container 内已经存在四个相互独立的文档。仅重置 positions 不会自动阻止 Attention 跨文档读取。


## 4. 每条样本的位置编号都从 0 重新开始

下面用两个很短的样本展示 packed representation。示例不需要发明特殊结束 token：第二次出现 position 0 的地方，就是第二条样本的开头。实际定长 container 若需要补齐，padding 区间也会单独从 position 0 开始。


In [ ]:
documents = [
    [11, 12, 13],
    [21, 22, 23, 24],
]

packed_tokens = [token for document in documents for token in document]
positions = [position for document in documents for position in range(len(document))]
document_ids = [doc_id for doc_id, document in enumerate(documents) for _ in document]

print('tokens:      ', packed_tokens)
print('positions:   ', positions)
print('document ids:', document_ids)


三组信息承担不同职责：

- **positions**：每条样本重新从 0 编号，同时让它的 RoPE 从 0 开始；
- **position 再次为 0 的位置**：让 trainer 恢复下一个隔离区间的起点；这些区间包括真实样本，也可能包括末尾 padding；
- **document ids / cumulative lengths**：由这些起点派生，让 Attention 判断 query 与 key 是否属于同一隔离区间。

重新编号本身不会修改 Attention mask；trainer 还要把这些起点转换成 block-causal mask 或累计长度。真实数据区间的起点对应的是**完整训练样本**，不是样本内部的每一条 chat message；补齐 container 时还会产生一个独立的 padding 区间。


## 5. Labels 与 padding

SFT labels 通常是右移一位的 token ids。Prompt token 与 padding token 对应的 label 会被替换为 `IGNORE_INDEX=-100`，从而不参与 loss。两件事：

- `label == -100` 控制该位置是否贡献训练目标；
- Attention mask 控制该位置可以读取哪些 K/V。

忽略某个位置的 loss，并不会阻止它被其他 token 读取。文档隔离仍必须由 Attention backend 实现。


## 6. 当前 `ChatDataset` 与 trainer 的真实路径

当前路径由上游 DataLoader 和 TorchTitan-NPU patch 共同完成：

```text
torchtitan/hf_datasets/text_datasets.py
  └─ ChatDataset._iter_greedy_packed()  # 输出 input / positions / labels
torchtitan_npu/patches/torchtitan/chat_dataset.py
  └─ 根据 attention 是否能隔离样本选择 greedy / non-greedy
torchtitan_npu/patches/torchtitan/trainer_post_dataloading_process.py
  └─ 从样本起点构造 block mask 或 VarlenMetadata
```

该方法会：

1. tokenization 后丢弃超过 `seq_len` 的样本；
2. 顺序填充 `_inputs_buffer` 与 `_labels_buffer`；
3. 在每条样本开头把 positions 重置为 0；
4. 放不下下一条样本时用 `eos_id` 和 ignored labels 补齐；
5. 输出固定长度的 `input`、`positions` 与 labels；
6. trainer 找到 positions 中所有重新从 0 开始的位置，恢复真实样本区间以及可能存在的末尾 padding 区间。

如果 attention 只有整段 causal 能力，patch 会关闭 greedy packing，让一条 sequence 至多包含一个真实样本；如果配置为 block-causal，才允许 greedy packing，并由 trainer 使用样本起点完成隔离。


### 6.1 Qwen multi-turn 的正确边界

Qwen3 tokenizer 的 `eos_id=151645` 对应 `<|im_end|>`，而 chat template 会在 user、assistant 等**每条消息**后写入 `<|im_end|>`。Wordle 又是一条样本包含多轮消息的任务。因此，扫描所有 `eos_id` 可能得到 message boundaries，而不是 sample boundaries；直接据此生成 Varlen segments 会阻止后续 assistant 读取同一局游戏的前文。

所以本教程使用的 SFT 训练路径只根据 position 重新从 0 开始的位置划分隔离区间，不读取 EOS。配套单测验证：同一样本内部的 EOS 不会产生新边界，两条 packed samples 之间仍然彼此不可见，末尾 padding 也被放进独立区间。


## 7. 当前 DataLoader 分流

DataLoader 初始化时会检查当前 attention 是否声明 `block_causal`。这决定能否安全地把多条样本放进同一个 container：

| Attention 能力 | Packing 策略 | 原因 |
|---|---|---|
| 只有整段 causal | non-greedy | 一个 sequence 只含一条样本，避免跨文档可见 |
| 支持 dense block-causal | greedy | 从 position 重置恢复样本区间并构造 boolean mask |
| 支持 Varlen metadata | greedy | 从同一批起点生成 `cu_seqlens`，由 TND backend 消费 |

实际行为可以概括为：

```python
supports_document_mask = attention.mask_type == 'block_causal'
greedy_packing = supports_document_mask
```

这段伪代码表达选择逻辑；真正判断由当前 ChatDataLoader patch 完成，不需要额外 CLI 参数。


## 8. 交给 Attention backend 的契约

DataLoader 与 Attention 的边界可以总结为：

```text
non-greedy
  └─ 每个 sequence 至多一个真实文档 → causal 足够

greedy
  └─ 一个 sequence 可含多个文档
       ├─ 每条样本的 positions 必须重新从 0 开始
       ├─ trainer 从这些起点恢复样本区间
       └─ Attention 必须支持 block-causal 或 Varlen
```

在进入性能比较前，至少应验证以下 invariants：container 长度固定、每条样本只出现一次、样本内 token 顺序不变、positions 正确重置、padding labels 为 `-100`、sample boundaries 可恢复，并且 multi-turn 消息不会被错误拆成独立文档。


## Packing 理解误区

- 把 container 数量当作原始样本数量；
- 用 `token != eos_id` 统计有效 token，因为真实样本本身也包含 EOS；
- 把 Qwen `<|im_end|>` 的每次出现都当成一条 Wordle 样本的结束；
- 认为 position 重置本身已经修改了 Attention mask；它只是保留起点，trainer 仍要生成 mask/metadata；
- 比较两条路线时只固定 `global_batch_size`，却不记录每个 step 实际包含多少原始样本。

端到端测量应使用 DataLoader 提供的样本起点或直接计数，不从 EOS 或 `labels != -100` 反推。


## 练习

1. （判断题）Greedy packing 提高 token-slot 利用率，但只重置 positions 并不会自动改变 Attention mask。

2. （单选题）当前 trainer 识别新隔离区间起点的依据是什么？
    A. 任意 EOS
    B. label 等于 IGNORE_INDEX
    C. 位置编号重新从 0 开始
    D. token id 等于 padding id

3. （多选题）positions、labels 和 padding 各自承担哪些职责？
    A. positions 的重新起算可表达隔离区间起点
    B. labels 决定哪些位置参与 loss
    C. padding 补齐 container，但不等于真实样本
    D. labels 会直接阻止跨样本 Attention

4. （判断题）若末尾 padding 自成一个隔离区间，则区间总数可能大于 raw sample 数。

In [ ]:
!cat ./answer/06.03_answer.txt
